Conda env uses conda-forge: python 3.15.2, pandas, geopandas, requests

References:
- https://www.eastbaytimes.com/2004/05/24/whats-in-a-street-name-new-guidebook-finds-out/
- https://geopandas.org/en/stable/getting_started/introduction.html#Making-maps 
- https://andreagordon.com/blog/the-secret-history-behind-berkeleys-street-names 
- https://berkeleyca.gov/city-services/community-gis-portal
- https://city-of-berkeley.hub.arcgis.com/ 
- https://gis.cityofberkeley.info/arcgis3/rest/services/Geocode/ 

The goal of this notebook is to review all location-dependent data for all datasets. This will allow me to map out necessary assumptions about the inputs and output for converting all location data to coordinate values.

In [2]:
import pandas as pd
from IPython.display import display

In [4]:
# loading in all data sets
PATH_TO_DATA = "../data/raw/"

files = {
    "cases": "Berkeley_PD_Cases_2016_to_Current.csv",
    "force": "use_of_force_2021_current.csv",
    "stops_1": "stop_data_2020_2023.csv",
    "stops_2": "stop_data_2024_current.csv",
}

data = {name : pd.read_csv(PATH_TO_DATA + filename) for name, filename in files.items()}

/tmp/ipykernel_74639/1044791719.py:11: DtypeWarning: Columns (12: IsK12School, 13: SchoolName, 14: StopForAStudent) have mixed types. Specify dtype option on import or set low_memory=False.
  data = {name : pd.read_csv(PATH_TO_DATA + filename) for name, filename in files.items()}


In [5]:
pd.DataFrame([{"dataset" : name, 
               "rows" : len(frame),
               "columns" : len(frame.columns)}
               for name, frame in data.items()])

,dataset,rows,columns
0,cases,146882,10
1,force,9933,18
2,stops_1,18090,44
3,stops_2,10931,58


In [ ]:
for name, frame in data.items():
    print(name)
    print(frame.columns.tolist())
    display(frame.head(5))

# there are so many different address names.... like
# cases: `Block_Address`
# force: `Block Address`
# stops_1: `LocationDescription`
# stops_2: a bunch of separate location fields...
# 
# Also I see `NonBerkeley Address` in force thats binary, no location specified. 
# and the force data shows a block wording repeated in kains street: 1200 Block of block of Kains

cases
['Case_Number', 'Occurred_Datetime', 'Incident_Type', 'Statute', 'Statute_Description', 'Statute_Type', 'Block_Address', 'City', 'ZIP_Code', 'ObjectId']


,Case_Number,Occurred_Datetime,Incident_Type,Statute,Statute_Description,Statute_Type,Block_Address,City,ZIP_Code,ObjectId
0,2025-00021341,2025/05/11 06:00:00+00,Disturbance,PC 602(I),TRESPASS: DESTROY FENCE,Penal Code,2100 CENTER ST,BERKELEY,94704,1
1,2021-00013841,2021/03/30 00:10:00+00,Theft Misdemeanor (Under $950),PC 484 (A),Petty Theft,Penal Code,2100 CENTER ST,BERKELEY,94704,2
2,2025-00017162,2025/04/17 03:05:00+00,Disturbance,PC 594(A)(1),VANDALISM:DEFACE PROPERTY,Penal Code,2100 SHATTUCK AVE,BERKELEY,94704,3
3,2021-00090675,2021/03/26 21:30:00+00,Burglary Commercial,PC 459,Burglary,Penal Code,2100 CENTER ST,BERKELEY,94704,4
4,2021-00013203,2021/03/26 18:30:00+00,Disturbance,PC 422,THREATEN CRIME W/INTENT TO TERRORIZE,Penal Code,2100 SHATTUCK AVE,BERKELEY,94704,5


force
['Incident Number', 'Subject_Incident Number', 'Sex', 'Race', 'Subject Role', 'Subject Age', 'Block Address', 'City', 'Use of Force Number', 'Officer_Incident Number', 'Officer Role', 'Use of Force Level', 'Use of Force Type', 'OccurredDate', 'Subject Arrested', 'Use of Force Reason', 'Subject Altered State', 'NonBerkeley Address']


,Incident Number,Subject_Incident Number,Sex,Race,Subject Role,Subject Age,Block Address,City,Use of Force Number,Officer_Incident Number,Officer Role,Use of Force Level,Use of Force Type,OccurredDate,Subject Arrested,Use of Force Reason,Subject Altered State,NonBerkeley Address
0,4475,3073.0,Male,Black,Arrested,45.0,700 Block of Folger,Berkeley,8647,10524,Cover Officer,Level 2,Level 2 Handgun DNP,2026-11-22,Yes,Reported as Armed,None detected,False
1,4475,3073.0,Male,Black,Arrested,45.0,700 Block of Folger,Berkeley,8646,10523,Primary Officer,Level 2,Level 2 Handgun Pointed,2026-11-22,Yes,Reported as Armed,None detected,False
2,4475,3072.0,Male,Black,Arrested,46.0,700 Block of Folger,Berkeley,8647,10524,Cover Officer,Level 2,Level 2 Handgun DNP,2026-11-22,Yes,Reported as Armed,None detected,False
3,4475,3072.0,Male,Black,Arrested,46.0,700 Block of Folger,Berkeley,8646,10523,Primary Officer,Level 2,Level 2 Handgun Pointed,2026-11-22,Yes,Reported as Armed,None detected,False
4,4468,1013.0,Male,Hispanic,Arrested,45.0,1200 Block of block of Kains,Berkeley,8636,10504,Primary Officer,Level 1,Level 1 Takedown,2026-11-16,Yes,Effect an Arrest,Unknown Drugs,False


stops_1
['LEA_RecordID', 'PersonNum', 'DateTimeOfStop', 'DateTimeSortable', 'DurationOfStop', 'RespToServiceCall', 'InfoBasedStop', 'TypeOfStop', 'LocationDescription', 'City', 'Latitude', 'Longitude', 'IsK12School', 'SchoolName', 'StopForAStudent', 'RacePerceivedPriorToStop', 'EthnicitySet', 'Ethnicity', 'Gender', 'GenderNonconforming', 'LGBT', 'Age', 'LimitedEnglish', 'DisabilitySet', 'Disability', 'CityOfResidence', 'PrimaryReasonForStop', 'TrafficViolationType', 'TrafficViolation', 'BasisOfSuspicion', 'SuspectedViolation', 'ActionsTaken', 'ConsentToSearchPerson', 'ConsentToSearchProperty', 'BasisForSearch', 'BasisForPropertySeizure', 'PropertySeized', 'ContrabandOrEvidence', 'WeaponsFound', 'ResultOfStop', 'WarningOffense', 'CitationForInfraction', 'InFieldCiteAndRelease', 'CustodialArrestNoWarrant']


,LEA_RecordID,PersonNum,DateTimeOfStop,DateTimeSortable,DurationOfStop,RespToServiceCall,InfoBasedStop,TypeOfStop,LocationDescription,City,...,BasisForSearch,BasisForPropertySeizure,PropertySeized,ContrabandOrEvidence,WeaponsFound,ResultOfStop,WarningOffense,CitationForInfraction,InFieldCiteAndRelease,CustodialArrestNoWarrant
0,BPD000021015,1,10/01/2020 00:13,20201001001300,15,No,NaN,Pedestrian,900 block of Allston Way,Berkeley,...,Condition of parole/probation/PRCS/mandatory s...,NaN,NaN,NaN,No,No action,NaN,NaN,NaN,NaN
1,BPD000021016,1,10/01/2020 03:00,20201001030000,25,Yes,NaN,Pedestrian,1800 block of San Pablo Ave,Berkeley,...,Condition of parole/probation/PRCS/mandatory s...,NaN,NaN,NaN,No,No action,NaN,NaN,NaN,NaN
2,BPD000021017,1,10/01/2020 04:58,20201001045800,45,Yes,NaN,Pedestrian,2100 block of Martin Luther King Jr Way,Berkeley,...,NaN,NaN,NaN,NaN,No,Psychiatric hold (W&I Code 5150 or 5585.20),NaN,NaN,NaN,NaN
3,BPD000021019,1,10/01/2020 04:59,20201001045900,16,Yes,NaN,Pedestrian,3100 block of College Ave,Berkeley,...,Officer safety/safety of others| Condition of ...,Contraband| Evidence,Other contraband or evidence,Other contraband or evidence,No,Custodial arrest without warrant,NaN,NaN,NaN,PC 459 BURGLARY:FIRST DEGREE (F)
4,BPD000021018,1,10/01/2020 05:05,20201001050500,24,No,NaN,Vehicle,3100 block Mabel St,Berkeley,...,Evidence of crime| Incident to arrest| Vehicle...,Evidence| Impound of vehicle,Firearms| Ammunition| Drugs/narcotics| Money| ...,Firearm(s)| Ammunition| Drugs/narcotics| Drug ...,Yes,Custodial arrest without warrant,NaN,NaN,NaN,HS 11378 POSS CNTL SUB FOR SALE (F)|PC 29800(A...


stops_2
['LEA_Record_ID', 'PersonNum', 'StopDate', 'StopTime', 'StopDuration', 'CallForService', 'WelfareWellnessCheck', 'InfoBasedStop', 'TypeOfStopDesc', 'LocationType', 'Latitude', 'Longitude', 'BlockNumber', 'StreetName', 'CrossStreet1', 'CrossStreet2', 'Highway', 'ClosestExit', 'OtherLocationDesc', 'City', 'IsK12School', 'SchoolName', 'StopForAStudent', 'RacePerceivedPriorToStop', 'PerceivedEthnicity', 'PerceivedGender', 'PerceivedNonbinaryPerson', 'PerceivedSexualOrientation', 'PerceivedAge', 'PerceivedLimitedEnglish', 'PerceivedDisability', 'PerceivedPersonUnhoused', 'PassengerInAVehicle', 'PersonInsideAResidence', 'CityOfResidence', 'TrafficViolationType', 'TrafficViolation', 'BasisOfSuspicion', 'SuspectedViolation', 'ProbableCause', 'ProbableCauseViolation', 'NonForceActionsTaken', 'ConsentToSearchPerson', 'ConsentToSearchProperty', 'BasisForSearch', 'ConsentForSearchKey', 'BasisForPropertySeizure', 'PropertySeized', 'WeaponsFound', 'ForceActionsTaken', 'ContrabandOrEvidence',

,LEA_Record_ID,PersonNum,StopDate,StopTime,StopDuration,CallForService,WelfareWellnessCheck,InfoBasedStop,TypeOfStopDesc,LocationType,...,WeaponsFound,ForceActionsTaken,ContrabandOrEvidence,ResultOfStop,VerbalWarningOffense,WrittenWarningOffense,CitationForInfractionOffense,InfieldCiteRelOffense,CustodialArrestNoWarrantOffense,Unnamed: 57
0,B00000000024,1,01/01/2024,01:20,10,Y,N,Y,Vehicular Stop,LatLonPt,...,No,NaN,NaN,Written warning,NaN,VC 22350 UNSAFE SPEED:PREVAIL COND (I),NaN,NaN,NaN,NaN
1,B00000000025,1,01/01/2024,03:22,25,Y,N,Y,Pedestrian Stop,LatLonPt,...,No,Handcuffed or flex cuffed,NaN,Verbal warning,PC 647(F) DISORD CONDUCT:ALCOHOL (M),NaN,NaN,NaN,NaN,NaN
2,B00000000031,1,01/01/2024,08:48,15,Y,N,N,Pedestrian Stop,LatLonPt,...,Yes,Handcuffed or flex cuffed,Weapon(s) other than a firearm,Psychiatric hold (W&I Code 5150 or 5585.20),NaN,NaN,NaN,NaN,NaN,NaN
3,B00000000026,1,01/01/2024,10:02,8,N,N,N,Vehicular Stop,LatLonPt,...,No,NaN,NaN,Citation for infraction,NaN,NaN,VC 22450(A) FAIL STOP VEH:XWALK/ETC (I),NaN,NaN,NaN
4,B00000000027,1,01/01/2024,10:20,8,N,N,N,Vehicular Stop,LatLonPt,...,No,NaN,NaN,Citation for infraction,NaN,NaN,VC 23123(A) USE CELLPH W/DRIV W/O HFD (I),NaN,NaN,NaN


In [9]:
# only looking into the location data for each dataset now 
location_data = {
    "cases" : ["Block_Address", "City", "ZIP_Code"],
    "force" : ["Block Address", "City", "NonBerkeley Address"],
    "stops_1" : ["Latitude", "Longitude", "LocationDescription", "City"],
    "stops_2" : ["LocationType", "Latitude", "Longitude", "BlockNumber", "StreetName", "CrossStreet1", "CrossStreet2", 
                 "Highway", "ClosestExit", "OtherLocationDesc", "City"]
}

# checking if there are some missing elements or columsn that may not be worth using as inputs
for name, columns in location_data.items():
    values = data[name][columns]
    print(name)
    display(pd.DataFrame({"missing" : values.isna().sum(), 
                         "unique counts" : values.nunique()}))
    display(values.sample(n=min(4, len(values)), random_state=64))

cases


,missing,unique counts
Block_Address,4155,4865
City,0,1
ZIP_Code,0,15


,Block_Address,City,ZIP_Code
114136,1200 EUCLID AVE,BERKELEY,94708
44208,3000 SHATTUCK AVE,BERKELEY,94705
138451,700 HILLDALE AVE,BERKELEY,94708
142161,3000 SAN PABLO AV,BERKELEY,94703


force


,missing,unique counts
Block Address,230,885
City,211,21
NonBerkeley Address,0,2


,Block Address,City,NonBerkeley Address
8241,NaN,NaN,True
8521,1600 Block of Oregon,Berkeley,False
8082,2300 Block of Piedmont,Berkeley,False
7827,1500 Block of Walnut,Berkeley,False


... (2 outputs left out)

,Latitude,Longitude,LocationDescription,City
4763,37.865,-122.298,700 block of Allston Way,Berkeley
10175,37.859,-122.280,Derby St,Berkeley
10627,37.881,-122.298,1200 block of 9th St,Berkeley
7863,37.869,-122.266,2100 block of Kittredge St,Berkeley


stops_2


,missing,unique counts
LocationType,0,5
Latitude,87,102
Longitude,87,107
BlockNumber,10884,24
StreetName,10884,45
CrossStreet1,10916,15
CrossStreet2,10916,15
Highway,10907,4
ClosestExit,10907,4
OtherLocationDesc,10930,1


,LocationType,Latitude,Longitude,BlockNumber,StreetName,CrossStreet1,CrossStreet2,Highway,ClosestExit,OtherLocationDesc,City
4371,LatLonPt,37.869,-122.294,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BERKELEY
8313,LatLonPt,37.878,-122.271,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BERKELEY
9681,LatLonPt,37.870,-122.281,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BERKELEY
6383,LatLonPt,37.859,-122.291,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BERKELEY


In [ ]:
# looks like stop data has coordinate data already. should check if these are consistent

for name in ["stops_1", "stops_2"]:
    frame = data[name]
    lat = pd.to_numeric(frame["Latitude"], errors="coerce")
    lon = pd.to_numeric(frame["Longitude"], errors="coerce")
    valid = (lat.between(-90, 90) & lon.between(-180, 180) & ~(lat.eq(0) & lon.eq(0)).fillna(False))

    print(name, {
        "valid_pair" : int(valid.sum()),
        "missing_invalid" : int((~valid).sum()) 
    })
    display(frame.loc[~valid, location_data[name]].head(15))

    # stops_1 has 14 bad pairs, stops_2 has 87 bad pairs... locationtype helps in stops_2 but not stops_1. 
    # stops_1 has location description for the cross street or the block number, ok to do case by case?
    # stops_2 have associated locationtype to inform the rule for obtaining the coordinates

stops_1 {'valid_pair': 18076, 'missing_invalid': 14}


,Latitude,Longitude,LocationDescription,City
15391,0.0,0.0,Ifo 1316 Gilman st,Berkeley
17250,0.0,0.0,2500 blk of Piedmont Ave,Berkeley
17496,0.0,0.0,2300 block of Ward Street,Berkeley
17510,0.0,0.0,2700 block of College Ave,Berkeley
17639,0.0,0.0,Allston Way/Milvia Street,Berkeley
17680,0.0,0.0,Durant/Shattuck,Berkeley
17765,0.0,0.0,2100 Martin Luther King Jr Way,Berkeley
17916,0.0,0.0,Kala Bagai and university,Berkeley
17965,0.0,0.0,3100 blk of Adeline Ave,Berkeley
17966,0.0,0.0,3200 blk of College Ave,Berkeley


stops_2 {'valid_pair': 10844, 'missing_invalid': 87}


,LocationType,Latitude,Longitude,BlockNumber,StreetName,CrossStreet1,CrossStreet2,Highway,ClosestExit,OtherLocationDesc,City
15,BlkNumber,NaN,NaN,1800.0,7th,NaN,NaN,NaN,NaN,NaN,BERKELEY
17,BlkNumber,NaN,NaN,2600.0,Shattuck Ave,NaN,NaN,NaN,NaN,NaN,BERKELEY
771,BlkNumber,NaN,NaN,2100.0,Martin Luther King Jr Way,NaN,NaN,NaN,NaN,NaN,BERKELEY
783,BlkNumber,NaN,NaN,1300.0,10th,NaN,NaN,NaN,NaN,NaN,BERKELEY
968,Intersection,NaN,NaN,NaN,NaN,Jones,Eastshore,NaN,NaN,NaN,BERKELEY
1076,BlkNumber,NaN,NaN,12800.0,San pablo Ave,NaN,NaN,NaN,NaN,NaN,BERKELEY
1285,BlkNumber,NaN,NaN,800.0,Hilldale,NaN,NaN,NaN,NaN,NaN,BERKELEY
1309,Intersection,NaN,NaN,NaN,NaN,A st,I880 W,NaN,NaN,NaN,SAN LORENZO
1311,HighwayAndExit,NaN,NaN,NaN,NaN,NaN,NaN,I880 W,A st,NaN,SAN LORENZO
1312,HighwayAndExit,NaN,NaN,NaN,NaN,NaN,NaN,880,Ast,NaN,HAYWARD


In [ ]:
# checking the text format for parsing rules
for name, column in [
    ("cases", "Block_Address"),
    ("force", "Block Address"),
    ("stops_1", "LocationDescription"),
]:
    print(name)
    display(data[name][column].value_counts(dropna=False).head(15))

display(data["stops_2"]["LocationType"].value_counts(dropna=False))

for name, frame in data.items():
    print(name)
    display(frame["City"].str.strip().str.upper().value_counts(dropna=False))

# many different interpretations found like:
# 1. approx block location
# 2. cross street intersection
# 3. numbered address
# 4. landmarks (need separate handling)
# 5. i see a unique situation where the text is obscured or something: `ASH00 AVE & WEST FRONTAGE RD`

cases


Block_Address
NaN                               4155
2100 SHATTUCK AVE                 3255
2100 MARTIN LUTHER KING JR WAY    2512
1400 UNIVERSITY AVE               1789
1300 SAN PABLO AVE                1628
1000 GILMAN ST                    1481
2600 TELEGRAPH AVE                1475
2800 ADELINE ST                   1466
2200 SHATTUCK AVE                 1120
2300 TELEGRAPH AVE                1110
1400 SHATTUCK AVE                 1039
2500 DURANT AVE                    981
1900 SHATTUCK AVE                  951
1800 UNIVERSITY AVE                882
2400 DURANT AVE                    814
Name: count, dtype: int64

force


Block Address
2100 Block of Martin Luther King Jr    357
NaN                                    230
2100 Block of Shattuck                 181
2400 Block of Ashby                    124
1800 Block of University               123
3100 Block of Harrison                 120
2200 Block of Shattuck                 113
2800 Block of College                  107
1900 Block of Center                    98
2600 Block of Telegraph                 97
2000 Block of Berkeley                  96
1400 Block of University                94
2000 Block of University                93
2800 Block of Adeline                   93
200 Block of Marina                     93
Name: count, dtype: int64

stops_1


LocationDescription
University Ave                             263
2100 block of Martin Luther King Jr Way    241
800 block of University Ave                189
2200 block of Shattuck Ave                 167
2600 block of Telegraph Ave                151
2000 block of University Ave               145
1400 block of University Ave               144
2100 block of Shattuck Ave                 141
1800 block of University Ave               132
1000 block of University Ave               130
1200 block of University Ave               129
San Pablo Ave                              129
1500 block of University Ave               126
1900 block of Shattuck Ave                 117
2500 block of Telegraph Ave                112
Name: count, dtype: int64

... (5 outputs left out)

stops_1


City
BERKELEY         17357
EMERYVILLE         317
OAKLAND            176
ALBANY             162
HAYWARD             31
RICHMOND            11
SAN FRANCISCO        6
SAN PABLO            5
EL CERRITO           4
ANTIOCH              3
ORINDA               2
VALLEJO              2
CONCORD              2
DUBLIN               1
CLAYTON              1
NOVATO               1
ELK GROVE            1
PINOLE               1
DALY CITY            1
HERCULES             1
LAFAYETTE            1
WALNUT CREEK         1
DIXON                1
FREMONT              1
MARTINEZ             1
Name: count, dtype: int64

stops_2


City
BERKELEY               10492
EMERYVILLE               145
OAKLAND                  129
ALBANY                    90
HAYWARD                   33
SAN LORENZO               12
UNION CITY                 8
ALAMEDA                    5
RICHMOND                   4
CASTRO VALLEY              4
EL CERRITO                 2
VALLEJO                    1
MARTINEZ                   1
ANTIOCH                    1
ANTELOPE                   1
SOUTH SAN FRANCISCO        1
TURLOCK                    1
SAN JOSE                   1
Name: count, dtype: int64

# Assumptions about the data
- Addresses containing numbers with `00` at the end of the number should be treated as block numbers. Ex. `"2100 SHATTUCK AVE"` will be treated as `"2100 BLOCK OF SHATTUCK AVE"`. 
- Separators between streets are to be treated as intersections.
- Coordinates that pass the range check will be treated as an adequate representation of the event location.
- The coordinates supplied by the Stop data are treated as World Geodetic System 1984 (WGS84) coordinates, using angular units where latitude and longitude in degrees. This will also be the chosen output convention (note that it is treated as interchangable with type EPSG registry code for the 2D geographic coordinate reference system (EPSG:4326)). 
- `(0,0)` is treated as a missing location in the datasets since it can't represent a Berkeley location.
- The `City` column supplies appropriate geocoding context, therefore the row is assumed to be within Berkeley's boundary. 

# Parsing rules for preprocessing 
- Missing values, empty strings, and whitespace strings treated as unavailable location components
- Standardizing strings by normalizing capitalization and removing extra whitespace
- Make all `BlockNumber` values integer. Zero is a valid block number
- Descriptions for locations involving blocks are converted to block number first then `BLOCK OF [STREET NAME]`. Repeated wording for block was found in an element for force data. Ex. `"1200 Block of block of Kains"` becomes `"1200 BLOCK OF KAINS"`. 
- Intersections names are characterized by their separators `/`, `&`, `and`. Ex. `"Allston Way/Milvia Street"` becomes `"ALLSTON WAY & MILVIA STREET"`
- Numbered addresses without block wording will be kept the same, just standardized by address number and all caps. 
- The newer Stop data, with `LocationType`, will determine the rules which describe the location. Specifically:
  - `LatLonPt` --> `Latitude` and `Longitude`.
  - `BlkNumber` --> `BlockNumber` and `StreetName`.
  - `Intersection` --> `CrossStreet1` and `CrossStreet2`.
  - `HighwayAndExit` --> highway/exit description requiring separate handling.
  - `Other` --> classify `OtherLocationDesc` using the text rules.
- Looking at location descriptions, if it contains the words `near`, `between`, `ifo` (I suspect meaning "in front of..."), the location will not be treated as an exact location description. They will remain unresolved by the initial automatic parsing since they describe proximity, not one point.
- Obscured or uncertain naming will be removed from analysis since we cannot reconstruct or replace them without obtaining more information.  

# Rules for converting to the output coordinates for all locations
-